# Problem 4 Implementing gradient computation via the multivariate chain rule (linear case)

In [25]:
import numpy as np

In [26]:
# 2. Implement a python function AffineTransformation(W, b, x) that can be used to compute either function by taking as input two arrays of parameters W and b and a vector x.

def AffineTransformation(W, b, x):
    """
    Computes the affine transformation of the input vector x using the weight matrix W and bias vector b.

    Parameters:
    W (numpy.ndarray): Weight matrix of shape (m, n) where m is the number of output features and n is the number of input features.
    b (numpy.ndarray): Bias vector of shape (m,) where m is the number of output features.
    x (numpy.ndarray): Input vector of shape (n,) where n is the number of input features.

    Returns:
    numpy.ndarray: The result of the affine transformation, which is a vector of shape (m,).
    """
    # Check if the dimensions of W, b, and x are compatible for the affine transformation
    if W.shape[1] != x.shape[0]:
        raise ValueError(f"Incompatible dimensions: W has shape {W.shape}, x has shape {x.shape}. The number of columns in W must match the number of rows in x.")
    if W.shape[0] != b.shape[0]:
        raise ValueError(f"Incompatible dimensions: W has shape {W.shape}, b has shape {b.shape}. The number of rows in W must match the number of rows in b.")
    return np.dot(W, x) + b


In [27]:
# Example usage input to function g:
W1 = np.array([[1, 2], [0, 1], [-1, 0]])
b1 = np.array([0, 0, 3])
x1 = np.array([-1, 1])
print(f"Array shape for example 1:\n W1: {W1.shape},\n b1: {b1.shape},\n x1: {x1.shape}")
result_g = AffineTransformation(W1, b1, x1)
print("Result of function g:", result_g, "with shape:", result_g.shape)
print()

# Example usage input to function f:
W2 = np.array([[1, -2, 1/4]])
b2 = np.array([0])
x2 = result_g  # Using the output of function g as input to function f
print(f"Array shapes for example 2:\n W2: {W2.shape},\n b2: {b2.shape},\n x2: {x2.shape}")
result_f = AffineTransformation(W2, b2, x2)
print("Result of function f:", result_f, "with shape:", result_f.shape)

Array shape for example 1:
 W1: (3, 2),
 b1: (3,),
 x1: (2,)
Result of function g: [1 1 4] with shape: (3,)

Array shapes for example 2:
 W2: (1, 3),
 b2: (1,),
 x2: (3,)
Result of function f: [0.] with shape: (1,)


In [28]:
# 3 Building on the previous function, implement a new function Composition(L, x) that calculates the composition of affine transformations represented as a list of pairs L= [(W1,b1),...,(Wn,bn)] and an initial vector x. Use the following convention: the functions are applied from the smallest to the largest index. Return all the activation values (i.e., intermediate results) as a list variable z = (z1,...,zn)
def Composition(L, x):
    """
    Computes the composition of affine transformations represented as a list of pairs (W, b) and an initial vector x.

    Parameters:
    L (list): A list of tuples where each tuple contains a weight matrix W and a bias vector b.
    x (numpy.ndarray): Initial input vector.

    Returns:
    list: A list of activation values (intermediate results) after applying each affine transformation.
    """
    z = []
    current_input = x
    for W, b in L:
        current_output = AffineTransformation(W, b, current_input)
        z.append(current_output)
        current_input = current_output  # Update the input for the next transformation
    return z

In [34]:
# 4 How would you call the previous function to compute (f◦g)(x), for x =[−1, 1]?
# Return the last outtput of the composition
L = [(W1, b1), (W2, b2)]
x_initial = np.array([-1, 1])
activation_values = Composition(L, x_initial)
print("\n Activation values for the composition (f◦g)(x):\n", activation_values,"\n")
print("\n Final Activation value for the composition (f◦g)(x):\n", activation_values[-1].item())


 Activation values for the composition (f◦g)(x):
 [array([1, 1, 4]), array([0.])] 


 Final Activation value for the composition (f◦g)(x):
 0.0


In [ ]:
# 8. Using the list of parameters L, the input vector x and all the activation values z obtained from the Composition, implement a function ComputeGradients(L,x,z) that returns the gradients of zn (ascalar) for all the parameters in L, i.e., ∇bi zn = (∂zn/∂[bi])⊤. and ∇Wi zn. To earn full marks,the solution should apply to any n≥2.

def ComputeGradients(L, x, z):
    """
    Computes the gradients of the final activation value zn with respect to all parameters in L.

    Parameters:
    L (list): A list of tuples where each tuple contains a weight matrix W and a bias vector b.
    x (numpy.ndarray): Initial input vector.
    z (list): A list of activation values (intermediate results) after applying each affine transformation.

    Returns:
    list: A list of tuples containing gradients for each layer, where each tuple contains (∇Wi zn, ∇bi zn).
    """
    gradients = []
    # Initialize the gradient of the final output with respect to itself
    grad_output = np.array([1.0])  # Since we are interested in the gradient of zn with respect to itself

    # Backpropagation through the layers
    for i in reversed(range(len(L))):
        W, b = L[i]
        current_z = z[i]
        
        # Compute gradients with respect to b
        grad_b = grad_output
        
        # Compute gradients with respect to W
        if i == 0:
            prev_z = x  # For the first layer, the input is x
        else:
            prev_z = z[i - 1]  # For subsequent layers, use the previous activation
        
        grad_W = np.outer(grad_output, prev_z)  # Outer product to get the gradient w.r.t W
        # W@x
        
        gradients.append((grad_W, grad_b))
        
        # Update grad_output for the next layer (backpropagation)
        grad_output = np.dot(W.T, grad_output)  # Propagate the gradient back through W

    gradients.reverse()  # Reverse to maintain the order of layers
    return gradients

In [31]:
# testing the ComputeGradients function
x1 = np.array([-1, 1])
L1 = [(W1, b1), (W2, b2)]
# Compute the activation values using the Composition function
z1 = Composition(L1, x1)
# Compute the gradients using the ComputeGradients function
gradients1 = ComputeGradients(L1, x1, z1)
# Print the gradients for each layer
for i, (grad_W, grad_b) in enumerate(gradients1):
    print(f"\nGradients for layer {i + 1}:")
    print("Gradient w.r.t W:\n", grad_W)
    print("Gradient w.r.t b:\n", grad_b)


Gradients for layer 1:
Gradient w.r.t W:
 [[-1.    1.  ]
 [ 2.   -2.  ]
 [-0.25  0.25]]
Gradient w.r.t b:
 [ 1.   -2.    0.25]

Gradients for layer 2:
Gradient w.r.t W:
 [[1. 1. 4.]]
Gradient w.r.t b:
 [1.]


In [32]:
# Testing another example with more layers
# Define weights and biases for a 3-layer network
# define new weights and biases for a 3-layer network
W1_new = np.array([[1, 2], [0, 1], [-1, 0], [2, -1], [1, 1]])
b1_new = np.array([0, 0, 3, 1, -1])
x1_new = np.array([-1, 1])
print(f"Array shape for example 1:\n W1_new: {W1_new.shape},\n b1_new: {b1_new.shape},\n x1_new: {x1_new.shape}")
result_g = AffineTransformation(W1_new, b1_new, x1_new)
print("Result of function g:", result_g, "with shape:", result_g.shape,"\n")

W2_new = np.array([[1, -2, 1/4, 0, 3], [0, 1, -1, 2, 1], [-1, 0, 2, -1, 0]])
b2_new = np.array([0, 1, 3])
x2_new = result_g  # Use the output of the first layer as input to the second layer
print(f"Array shapes for example 2:\n W2_new: {W2_new.shape},\n b2_new: {b2_new.shape},\n x2_new: {x2_new.shape}")
result_f = AffineTransformation(W2_new, b2_new, x2_new)
print("Result of function f:", result_f, "with shape:", result_f.shape,"\n")

W3_new = np.array([[1, 0, -1]])
b3_new = np.array([2])
x3_new = result_f  # Use the output of the second layer as input to the third layer
print(f"Array shapes for example 3:\n W3_new: {W3_new.shape},\n b3_new: {b3_new.shape},\n x3_new: {x3_new.shape}")
result_h = AffineTransformation(W3_new, b3_new, x3_new)
print("Result of function h:", result_h, "with shape:", result_h.shape,"\n")


Array shape for example 1:
 W1_new: (5, 2),
 b1_new: (5,),
 x1_new: (2,)
Result of function g: [ 1  1  4 -2 -1] with shape: (5,) 

Array shapes for example 2:
 W2_new: (3, 5),
 b2_new: (3,),
 x2_new: (5,)
Result of function f: [-3. -7. 12.] with shape: (3,) 

Array shapes for example 3:
 W3_new: (1, 3),
 b3_new: (1,),
 x3_new: (3,)
Result of function h: [-13.] with shape: (1,) 



In [33]:

L2 = [(W1_new, b1_new), (W2_new, b2_new), (W3_new, b3_new)]
# compute the activation values using the Composition function
z2 = Composition(L2, x1_new)
# compute the gradients using the ComputeGradients function
gradients2 = ComputeGradients(L2, x1_new, z2)
# Print the gradients for each layer
for i, (grad_W, grad_b) in enumerate(gradients2):
    print(f"\nGradients for layer {i + 1} in the 3-layer network:")
    print("Gradient w.r.t W:\n", grad_W)
    print("Gradient w.r.t b:\n", grad_b)


Gradients for layer 1 in the 3-layer network:
Gradient w.r.t W:
 [[-2.    2.  ]
 [ 2.   -2.  ]
 [ 1.75 -1.75]
 [-1.    1.  ]
 [-3.    3.  ]]
Gradient w.r.t b:
 [ 2.   -2.   -1.75  1.    3.  ]

Gradients for layer 2 in the 3-layer network:
Gradient w.r.t W:
 [[ 1.  1.  4. -2. -1.]
 [ 0.  0.  0. -0. -0.]
 [-1. -1. -4.  2.  1.]]
Gradient w.r.t b:
 [ 1.  0. -1.]

Gradients for layer 3 in the 3-layer network:
Gradient w.r.t W:
 [[-3. -7. 12.]]
Gradient w.r.t b:
 [1.]
